# Econ 4850 · Homework 5: Simulated method of moments

Geoffrey Kam, Washington University in St. Louis · Spring 2025

This homework asks you to estimate a linear regression with normal errors by the simulated method of moments (SMM). Keep in mind that SMM is much more flexible than this simple case.

Answer every task and question in this notebook. Put your code in code cells and your written answers and explanations in markdown cells. The marks for each item are shown in its heading.

## Step 0: The model

We consider the multiple linear regression model

$$
y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \epsilon_i, \quad i = 1, \dots, N,
$$

where

- $(x_{i1}, x_{i2})' \sim \mathcal{N}\big((0, 0)', I_2\big)$ (exogenous regressors);
- $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$ (i.i.d. normal errors);
- the true parameters are
  - $\beta_0 = 1$,
  - $\beta_1 = 2$,
  - $\beta_2 = -1$,
  - $\sigma^2 = 1$;
- the sample size is $N = 1000$. Use this value throughout.

## Step 1: Simulating the data

### Task 0: Write a function to simulate data

Define a function `simulate_data(N, beta0, beta1, beta2, sigma)` that:

- takes $N$ (the sample size), $\beta_0$, $\beta_1$, $\beta_2$ and $\sigma$ as inputs;
- generates a dataset of $N$ observations $\{(x_{i1}, x_{i2}, y_i)\}_{i=1}^{N}$;
- saves the data in a pandas DataFrame with column names `['y', 'x1', 'x2']`;
- returns the DataFrame;
- uses `np.random.seed(850)`.

This function is provided in the code cell below; the remaining tasks build on it.

In [ ]:
import numpy as np
import pandas as pd


def simulate_data(N, beta0, beta1, beta2, sigma):
    np.random.seed(850)  # set seed for reproducibility
    X = np.random.normal(0, 1, (N, 2))  # two regressors
    epsilon = np.random.normal(0, sigma, N)
    y = beta0 + beta1 * X[:, 0] + beta2 * X[:, 1] + epsilon

    # Store in a DataFrame
    data = pd.DataFrame({'y': y, 'x1': X[:, 0], 'x2': X[:, 1]})
    return data

### Question 1: Reproducibility (1 mark)

The provided function calls `np.random.seed(850)` inside its body rather than once at the top of the notebook. Why? What could go wrong if the seed were set only once at the top?

## Step 2: Calculate the target moments

SMM minimizes the distance between the moments implied by the simulated model and the empirical moments (here, the moments of the dataset you simulated in Step 1). We therefore first have to calculate the moments from the data.

### Question 2: Choosing the moments (1 mark)

Using your intuition from OLS, which moments would you include? Why?

Hint: recall the closed-form solution for the OLS coefficients. Note that you estimate four parameters, $(\beta_0, \beta_1, \beta_2, \sigma)$, so you need at least four moments, and at least one of them must be informative about $\sigma$; the OLS formula alone does not involve $\sigma$.

### Task 1: Moment function (1 mark)

Write a function that computes the moments you suggested above from a dataset. Make use of pandas aggregation methods.

## Step 3: Compute the weighting matrix

Because we compare several moments at once, and their variability can differ across moments, we weight them using the inverse of the covariance matrix of the moment conditions $g(\theta)$. However, that matrix is a function of $\theta$, the unknown parameter vector we are trying to estimate.

There are several ways to approach this problem. Here, we simply bootstrap the covariance matrix of the empirical moments.

### Task 2: Bootstrap weighting matrix (1 mark)

Bootstrap the covariance matrix of your **empirical moments**: resample the data with `sample(frac=1, replace=True)`, using `B=500` bootstrap samples, and compute the covariance matrix of the bootstrapped moments (you may use `np.cov`). Write this as a function whose inputs are `(empirical_data, moment_function, B=500)`. Use the pseudo-inverse rather than the ordinary inverse of the covariance matrix, for numerical stability.

## Step 4: Simulate the model

Here we assume the model is a linear model with normal errors. However, $y$ depends on $\epsilon$, which is not observed in the data, so we have to simulate $\epsilon$ in order to obtain $y$. Fix the seed of your simulated errors; in other words, use the "same" set of errors throughout the minimization.

Note: $\sigma$ is a standard deviation, so it must stay positive. Impose the bound $\sigma > 0$ when you minimize the objective in Step 6.

Note: you must use a set of errors different from the one used to generate the data in Step 1, because in reality there is no way to know the values of the errors. Hence, call `np.random.seed(55555)` inside your simulation function, immediately before drawing the errors, so that every evaluation of the objective uses the same draws.

### Task 3: Simulation function (1 mark)

Write a function that simulates a set of errors of the same size as the sample, $N$, and uses them to simulate $y$. The parameters $\beta_0$, $\beta_1$, $\beta_2$ and $\sigma$ must be inputs of this function, because we will later minimize the objective over them.

## Step 5: Write down the objective

### Task 4: Objective function (1 mark)

Let $\theta = (\beta_0, \beta_1, \beta_2, \sigma)$. Write a function for the objective

$$
\big(\hat{g}(\theta, \text{data}) - g(\text{data})\big)' \, W \, \big(\hat{g}(\theta, \text{data}) - g(\text{data})\big),
$$

where $g(\text{data})$ is the vector of empirical moments, $\hat{g}(\theta, \text{data})$ is the vector of moments of the data simulated at $\theta$, and $W$ is the weighting matrix. The difference $\hat{g}(\theta, \text{data}) - g(\text{data})$ is an $M \times 1$ vector, where $M$ is the number of moments you target. The arguments of this function should be `(theta, X, data_moments, moment_function, weighting_matrix)`.

## Step 6: Minimize the objective

### Task 5: Estimation (1 mark)

Use your favorite minimization package to minimize the objective over $\beta_0$, $\beta_1$, $\beta_2$ and $\sigma$; for example, `scipy.optimize.minimize`. Since this problem is not too complex, `method='L-BFGS-B'` should work (you may run into an error; try to understand it and overcome it). You may use other methods too. You must use the initial guess `np.array([0, 0, 0, 2])`, and impose $\sigma > 0$ (with L-BFGS-B, through the `bounds` argument).

## Checking your answer

Because the true parameters are known, you can check your answer easily.

### Task 6: Comparison with OLS (1 mark)

Use the closed-form OLS estimator to compute the $\beta$ coefficients, and compare them with your SMM estimates.

Note: there is nothing we can do about the unobserved $\epsilon$.